In [32]:
import os
os.environ['HADOOP_CONF_DIR'] = '/etc/hadoop/conf'
os.environ['YARN_CONF_DIR'] = '/etc/hadoop/conf'

import findspark
findspark.init()
findspark.find()

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("3")\
        .master("yarn")\
        .config("spark.executor.memory", "4g") \
        .config("'spark.executor.cores", "4")\
        .config("'spark.driver.cores", "4")\
        .config("spark.ui.port", "4051")\
        .getOrCreate()

#проверка

events = spark.read.json(path = "/user/master/data/events/date=2022-05-25")

events.show(10)




2025-03-25 12:47:06,244 WARN util.Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
2025-03-25 12:47:12,747 WARN util.Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
2025-03-25 12:47:12,750 WARN cluster.YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!
2025-03-25 12:47:17,129 WARN cluster.YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 1 for reason Container from a bad node: container_1729228596238_26237_01_000002 on host: rc1a-dataproc-d-5w4oxa2s5b8foehs.mdb.yandexcloud.net. Exit status: 137. Diagnostics: [2025-03-25 12:47:15.043]Container killed on request. Exit code is 137
[2025-03-25 12:47:15.043]Container exited with a non-zero exit code 137. 
[2025-03-25 12:47:15.043]Killed by external signal
.


+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|[,, 2022-05-25 17...|subscription|
|[,, 2022-05-25 15...|subscription|
|[,, 2022-05-25 18...|subscription|
|[,, 2022-05-25 00...|subscription|
|[,, 2022-05-25 03...|subscription|
|[,, 2022-05-25 04...|subscription|
|[,, 2022-05-25 15...|subscription|
|[,, 2022-05-25 01...|subscription|
|[,, 2022-05-25 00...|subscription|
|[,, 2022-05-25 07...|subscription|
+--------------------+------------+


In [1]:


from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("3")\
        .master("yarn")\
        .config("spark.executor.memory", "4g") \
        .config("'spark.executor.cores", "4")\
        .config("'spark.driver.cores", "4")\
        .config("spark.ui.port", "4051")\
        .getOrCreate()

#проверка

events = spark.read.json(path = "/user/master/data/events/date=2022-05-25")

events.show(10)

/opt/spark/conf/spark-env.sh: line 31: hadoop: command not found


26/06/05 12:46:10 WARN Utils: Your hostname, fv4l76n2jdl2n7frsa5i resolves to a loopback address: 127.0.1.1; using 10.130.0.9 instead (on interface eth0)
26/06/05 12:46:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/05 12:46:12 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/06/05 12:46:22 WARN Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your configs.
26/06/05 12:46:22 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


+--------------------+------------+
|               event|  event_type|
+--------------------+------------+
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
|{null, null, 2022...|subscription|
+--------------------+------------+
only showing top 10 rows



In [2]:
events = spark.read.json("/user/master/data/events")
events.write \
        .partitionBy("date", "event_type") \
        .mode("overwrite") \
        .parquet("/user/s28509656/data/events")

ЗАДАНИЕ 5.2.4

In [ ]:
import sys


import pyspark.sql.functions as F
import datetime




def find_candidates(messages, verified_tags, suggested_cutoff):
        all_tags = messages\
        .where("event.message_channel_to is not null")\
        .selectExpr(["event.message_from as user", "explode(event.tags) as tag"])\
        .groupBy("tag")\
        .agg(F.countDistinct("user").alias("suggested_count"))\
        .where(F.col("suggested_count") >= suggested_cutoff)

        return all_tags.join(verified_tags, "tag", "left_anti")
    
    


date = "2022-05-31"
days_count = 5
suggested_cutoff = 300
base_input_path = "/user/vasilyevol/data/events"
verified_tags_path = "/user/master/data/snapshots/tags_verified/actual"
base_output_path = "/user/vasilyevol/5.2.4/analytics/verified_tags_candidates_d5"



dt = datetime.datetime.strptime(date, '%Y-%m-%d')
paths = [f"{base_input_path}/date={(dt-datetime.timedelta(days=x)).strftime('%Y-%m-%d')}/event_type=message" for x in range(int(days_count))]
messages = spark.read.parquet(*paths)

verified_tags = spark.read.parquet(verified_tags_path)

candidates = find_candidates(messages, verified_tags, suggested_cutoff)

candidates.write.parquet(f"{base_output_path}/date={date}")


In [ ]:
!hadoop fs -ls /user/vasilyevol/data/analytics/candidates_d7_pyspark


5.3.1

In [5]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def input_event_paths(date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"/user/s28509656/data/events/date={(dt - datetime.timedelta(days=x)).strftime('%Y-%m-%d')}" for x in
            range(depth)]


def tag_tops(date, depth, spark):
    message_paths = input_event_paths(date, depth)
    message_paths = input_event_paths(date, depth)
    result = spark.read\
    .option("basePath", "/user/s28509656/data/events")\
    .parquet(*message_paths)\
    .where("event_type = 'message'")\
    .where("event.message_channel_to is not null")\
    .select(F.col("event.message_id").alias("message_id"),
            F.col("event.message_from").alias("user_id"),
            F.explode(F.col("event.tags")).alias("tag"))\
    .groupBy("user_id", "tag")\
    .agg(F.count("*").alias("tag_count"))\
    .withColumn("rank", F.row_number().over(Window.partitionBy("user_id")\
    .orderBy(F.desc("tag_count"), F.desc("tag"))))\
    .where("rank <= 3")\
    .groupBy("user_id")\
    .pivot("rank", [1, 2, 3])\
    .agg(F.first("tag"))\
    .withColumnRenamed("1", "tag_top_1")\
    .withColumnRenamed("2", "tag_top_2")\
    .withColumnRenamed("3", "tag_top_3")
    result.write.mode("overwrite").parquet(f"/user/s28509656/data/tmp/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")

    return None








In [6]:
depths=[5,5,1]
dates=['2022-06-04', '2022-05-04', '2022-05-04']

# dates=[ '2022-04-04', '2022-04-04']

s = zip(dates,depths)
for date,depth in s:
    tag_tops(date, depth, spark)

5.3.2

In [7]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window


def input_event_paths(date, depth):
    dt = datetime.datetime.strptime(date, '%Y-%m-%d')
    return [f"/user/s28509656/data/events/date={(dt - datetime.timedelta(days=x)).strftime('%Y-%m-%d')}" for x in
            range(depth)]


def reaction_tag_tops(date, depth, spark):
    reaction_paths = input_event_paths(date, depth)
    reactions = spark.read\
    .option("basePath", "/user/s28509656/data/events")\
    .parquet(*reaction_paths)\
    .where("event_type='reaction'")

    all_message_tags = spark.read.parquet("/user/s28509656/data/events")\
    .where("event_type='message' and event.message_channel_to is not null")\
    .select(F.col("event.message_id").alias("message_id"),
            F.col("event.message_from").alias("user_id"),
            F.explode(F.col("event.tags")).alias("tag")
    )

    reaction_tags = reactions\
    .select(F.col("event.reaction_from").alias("user_id"), 
            F.col("event.message_id").alias("message_id"), 
            F.col("event.reaction_type").alias("reaction_type")
    ).join(all_message_tags.select("message_id", "tag"), "message_id")

    reaction_tops = reaction_tags\
    .groupBy("user_id", "tag", "reaction_type")\
    .agg(F.count("*").alias("tag_count"))\
    .withColumn("rank", F.row_number().over(Window.partitionBy("user_id", "reaction_type")\
    .orderBy(F.desc("tag_count"), F.desc("tag"))))\
    .where("rank <= 3")\
    .groupBy("user_id", "reaction_type")\
    .pivot("rank", [1, 2, 3])\
    .agg(F.first("tag"))\
    .cache()

    like_tops = reaction_tops\
    .where("reaction_type = 'like'")\
    .drop("reaction_type")\
    .withColumnRenamed("1", "like_tag_top_1")\
    .withColumnRenamed("2", "like_tag_top_2")\
    .withColumnRenamed("3", "like_tag_top_3")

    dislike_tops = reaction_tops\
    .where("reaction_type = 'dislike'")\
    .drop("reaction_type")\
    .withColumnRenamed("1", "dislike_tag_top_1")\
    .withColumnRenamed("2", "dislike_tag_top_2")\
    .withColumnRenamed("3", "dislike_tag_top_3")

    result = like_tops\
    .join(dislike_tops, "user_id", "full_outer")
    print("here 1 ")
    result.write.mode("overwrite").parquet(f"/user/s28509656/data/tmp/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    return None




In [8]:
depths=[5,5,1]
dates=['2022-05-04', '2022-04-04', '2022-04-04']
s = zip(dates,depths)
for date,depth in s:
    reaction_tag_tops(date, depth, spark)

here 1 


here 1 


here 1 


5.3.3

In [10]:
depths=[5,5,1]
dates=['2022-04-04', '2022-04-05', '2022-04-04']
# dates=[ '2022-04-04', '2022-04-04']

s = zip(dates,depths)
for date,depth in s:
    tag_tops(date, depth, spark)

In [13]:
import datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window
depths=[5,5,1]
dates=['2022-05-04', '2022-04-04', '2022-04-04']
s = zip(dates,depths)

def calculate_user_interests(date, depth, spark):
    print(f"/user/s28509656/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops = spark.read.parquet(f"/user/s28509656/data/tmp/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    reaction_tops = spark.read.parquet(f"/user/s28509656/data/tmp/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops.join(reaction_tops, "user_id", "full_outer").write.mode("overwrite").\
    parquet(f"/user/s28509656/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    
    
for date,depth in s:
    calculate_user_interests(date, depth, spark)



/user/s28509656/data/tmp/user_interests_05_04_5


/user/s28509656/data/tmp/user_interests_04_04_5


/user/s28509656/data/tmp/user_interests_04_04_1


In [11]:
def calculate_user_interests(date, depth, spark):
    print(f"/user/vasilyevol/data/tmp/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops = spark.read.parquet(f"/tmp/data/tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    reaction_tops = spark.read.parquet(f"/tmp/data/reaction_tag_tops_{date[5:7]}_{date[8:10]}_{depth}")
    post_tops.join(reaction_tops, "user_id", "full_outer").write.mode("overwrite").\
    parquet(f"/tmp/data/user_interests_{date[5:7]}_{date[8:10]}_{depth}")
    
    
for date,depth in s:
    calculate_user_interests(date, depth, spark)

In [12]:
depths=[5,5,1]
dates=['2022-04-04', '2022-04-05', '2022-04-04']
for date,depth in s:
    calculate_user_interests(date, depth, spark)